# 🖼️ AI Image Captioner — Web Scraping & Multi-Source Vision AI

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mitadrudeb/building-genai-powered-applications/blob/main/01-ai-image-captioner/AI_Image_Captioner.ipynb)


**Author:** Mitadru Deb  
**Role Focus:** Beginner LLM & Multimodal AI Engineering  
**Model:** [Salesforce/blip-image-captioning-base](https://huggingface.co/Salesforce/blip-image-captioning-base) (~0.25B parameters)  
**Last Updated:** November 2026

---

## 📌 Project Overview

Welcome to the **AI Image Captioner** project! This project demonstrates how to merge **computer vision**, **natural language processing (NLP)**, and **real-world web scraping** into a single interactive web application.

The application provides two core capabilities:
1. **📤 Single Image Upload:** Upload any image from your computer or phone to receive a descriptive AI caption.
2. **🌐 Webpage URL Auto-Captioning:** Enter any webpage link (e.g. Wikipedia articles, blogs, news). The app scrapes the HTML, discovers all images, filters out tiny icons/SVGs, downloads valid photos, and automatically generates captions for every image found.

### 🤖 Model Details: Salesforce BLIP
- **Parameters:** ~247M (~0.25 Billion) — ultra-lightweight and efficient.
- **Architecture:** Bootstrapping Language-Image Pre-training (BLIP). Uses a Vision Transformer (**ViT-B/16**) image encoder coupled with a **BERT-based text decoder** via cross-attention.
- **Hardware Requirement:** Runs smoothly on standard consumer **CPUs**, and automatically utilizes **NVIDIA CUDA GPUs** when available for fast inference.
- **Compatibility:** Runs on **Google Colab, Windows, Linux, and macOS** with zero API keys and zero cost.

---

## 📑 Table of Contents

1. [Setup & Installation](#1-setup--installation)
2. [Import Libraries & Environment Check](#2-import-libraries--environment-check)
3. [Load the Pretrained BLIP Model](#3-load-the-pretrained-blip-model)
4. [Core Captioning Pipeline](#4-core-captioning-pipeline)
   - 4a. Single Image Captioning
   - 4b. Web Scraping & Automated URL Captioning
5. [Launch the Gradio Web Application](#5-launch-the-gradio-web-application)
6. [Engineering Takeaways & Portfolio Notes](#6-engineering-takeaways--portfolio-notes)

---

## 1. Setup & Installation

Run this cell once to install all necessary packages.  
- On **Google Colab**, this installs everything into your temporary cloud environment.
- Locally, you can also run: `pip install -r requirements.txt`

In [ ]:
# Install core dependencies
!pip install -q transformers torch torchvision pillow gradio requests beautifulsoup4

---

## 2. Import Libraries & Environment Check

We import the foundational tools of modern AI engineering:
- **`torch` & `torchvision`**: Deep learning backend for tensor computations and device management.
- **`transformers`**: Hugging Face library to download model weights and processors.
- **`PIL` (Pillow)**: Image loading, format conversion (RGBA/Greyscale → RGB), and geometry inspection.
- **`requests` & `bs4` (BeautifulSoup)**: Robust HTTP fetching and HTML parsing for automated image extraction.
- **`gradio`**: Web UI framework to build clean, shareable applications.

In [ ]:
import os
import platform
from io import BytesIO
from urllib.parse import urljoin

import requests
import numpy as np
from PIL import Image
from bs4 import BeautifulSoup
import gradio as gr

import torch
from transformers import AutoProcessor, BlipForConditionalGeneration

# Verify the runtime environment
print(f"Operating System: {platform.system()} {platform.release()}")
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Detected:     {torch.cuda.get_device_name(0)}")
else:
    print("Execution Mode:   CPU (Standard)")

---

## 3. Load the Pretrained BLIP Model

We load **`Salesforce/blip-image-captioning-base`** directly from the Hugging Face Model Hub.

### 🔍 How the Vision-to-Text Pipeline Works

```
┌──────────────┐     ┌──────────────────────┐     ┌─────────────────────┐     ┌──────────────┐
│  Input Image │ ──► │    AutoProcessor     │ ──► │  BLIP Vision Model  │ ──► │ Text Caption │
│  (RGB Array) │     │ (Resize & Normalize) │     │ (ViT Encoder + BERT)│     │  (Decoded)   │
└──────────────┘     └──────────────────────┘     └─────────────────────┘     └──────────────┘
```

1. **`AutoProcessor`**: Handles image preprocessing (resizing to 384x384, normalizing pixel channels) and tokenizes text prompts.
2. **`BlipForConditionalGeneration`**: Takes the image visual tokens and generates the most probable sequence of descriptive words.

In [ ]:
# Configure precision and execution device
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Target Device: {device}")
print(f"Target Dtype:  {torch_dtype}")
print()

print("Loading Salesforce BLIP processor and model...")
processor = AutoProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base",
    torch_dtype=torch_dtype,
).to(device)

print("✅ Model and processor loaded successfully into memory!")

---

## 4. Core Captioning Pipeline

We construct clean, modular functions for both inference paths:
- `generate_caption()`: Internal helper that runs tensor transformation and text decoding.
- `caption_image()`: UI handler for uploaded user images.
- `caption_from_url()`: Web scraper and batch captioner.

### 4a. Single Image Captioning

The `generate_caption` function accepts a PIL Image, pre-prompts the model with `"the image of"` to guide descriptive generation, and returns decoded human-readable text.

In [ ]:
def generate_caption(pil_image: Image.Image) -> str:
    """
    Run inference on a single PIL image using BLIP.
    """
    # Pass image and prompt to the processor
    inputs = processor(
        images=pil_image,
        text="the image of",
        return_tensors="pt",
    ).to(device)

    # If using GPU, cast pixel tensor to float16 for efficiency
    if device == "cuda":
        inputs["pixel_values"] = inputs["pixel_values"].to(torch_dtype)

    # Generate token sequence
    output_tokens = model.generate(**inputs, max_new_tokens=50)

    # Decode tokens into text
    caption = processor.decode(output_tokens[0], skip_special_tokens=True)
    return caption


def caption_image(input_image: np.ndarray) -> str:
    """
    Callback for the 'Upload Image' tab.
    Converts Gradio's NumPy array into a PIL RGB Image.
    """
    if input_image is None:
        return "⚠️ Please upload an image first."

    pil_image = Image.fromarray(input_image).convert("RGB")
    return generate_caption(pil_image)


print("✅ caption_image() defined.")

### 4b. Web Scraping & Automated URL Captioning

This function implements real-world data ingestion:
1. **HTTP Request**: Sends a GET request with standard browser headers to prevent basic bot blocks.
2. **HTML Parsing**: Uses `BeautifulSoup` to find all `<img>` tags (`src`, `data-src`, or `srcset`).
3. **Filtering**: Automatically ignores vector files (`.svg`), base64 data URIs, and tiny tracking pixels/icons (`< 400px` area).
4. **URL Resolution**: Uses `urllib.parse.urljoin` to turn relative links (`//upload.wikimedia...` or `/images/...`) into absolute URLs.
5. **Batch Inference**: Downloads each valid image in-memory via `BytesIO` and generates a caption.

In [ ]:
def caption_from_url(url: str, max_images: int = 5) -> str:
    """
    Scrape images from a webpage URL and caption each valid image.

    Args:
        url:         Webpage URL (e.g. https://en.wikipedia.org/wiki/Computer)
        max_images:  Maximum number of images to caption (default: 5)

    Returns:
        Formatted string listing each image URL and its AI caption.
    """
    if not url or not url.strip():
        return "⚠️ Please enter a valid URL."

    url = url.strip()
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    # Step 1: Fetch HTML content
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
    except requests.RequestException as e:
        return f"❌ Could not fetch webpage: {e}"

    # Step 2: Extract all <img> tags
    soup = BeautifulSoup(response.text, "html.parser")
    img_elements = soup.find_all("img")

    if not img_elements:
        return "No images found on this page."

    # Step 3: Iterate, filter, download, and caption
    results = []
    captioned_count = 0

    for idx, img_element in enumerate(img_elements, start=1):
        if captioned_count >= max_images:
            break

        # Extract image URL from common attributes
        img_url = img_element.get("src") or img_element.get("data-src")
        if not img_url and img_element.has_attr("srcset"):
            img_url = img_element["srcset"].split()[0]
        if not img_url:
            continue

        # Skip SVGs (vector graphics) and base64 data URIs
        if ".svg" in img_url.lower() or img_url.startswith("data:"):
            continue

        # Convert relative paths to valid absolute URLs
        img_url = urljoin(url, img_url)
        if not img_url.startswith("http"):
            continue

        # Download the image and generate caption
        try:
            img_response = requests.get(img_url, timeout=10, headers=headers)
            pil_image = Image.open(BytesIO(img_response.content))

            # Filter out tiny icon / tracking pixels
            width, height = pil_image.size
            if width * height < 400 or width < 30 or height < 30:
                continue

            pil_image = pil_image.convert("RGB")
            caption = generate_caption(pil_image)

            captioned_count += 1
            results.append(
                f"📷 Image {captioned_count}:\n"
                f"   URL: {img_url}\n"
                f"   Caption: {caption}\n"
            )

        except Exception:
            # Skip corrupted or inaccessible image files
            continue

    if not results:
        return "No captionable images found (images may be SVGs, too small, or blocked)."

    header = (
        f"✅ Captioned {captioned_count} image(s) from: {url}\n"
        f"Model: Salesforce BLIP (~0.25B parameters)\n"
        f"{'=' * 60}\n\n"
    )
    return header + "\n".join(results)


print("✅ caption_from_url() defined.")

---

## 5. Launch the Gradio Web Application

We construct an interactive, dual-tab web application using **`gr.Blocks`**:
- **Tab 1: 📤 Upload Image** — Upload any local photo and get an instant caption.
- **Tab 2: 🌐 Caption from URL** — Input a URL and adjust the slider to control how many images are scraped and captioned.

> 🔗 **Colab Sharing:** In Google Colab, `app.launch(share=True)` automatically generates a public `.gradio.live` link so anyone can try your app live!

In [ ]:
# Detect if executing in Google Colab
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

with gr.Blocks(title="AI Image Captioner") as app:

    gr.Markdown(
        """
        # 🖼️ AI Image Captioner
        Generate image captions using the open-source **Salesforce BLIP** vision-language model.  
        **Upload an image** or **paste a webpage URL** to automatically describe images.
        """
    )

    with gr.Tabs():

        # ---- Tab 1: Upload Image ----
        with gr.TabItem("📤 Upload Image"):
            with gr.Row():
                with gr.Column():
                    upload_image = gr.Image(
                        type="numpy",
                        label="Upload an Image",
                    )
                    upload_btn = gr.Button("Generate Caption", variant="primary")

                with gr.Column():
                    upload_output = gr.Textbox(
                        label="Generated Caption",
                        lines=5,
                    )

            upload_btn.click(
                fn=caption_image,
                inputs=upload_image,
                outputs=upload_output,
            )

        # ---- Tab 2: Caption from URL ----
        with gr.TabItem("🌐 Caption from URL"):
            with gr.Row():
                with gr.Column():
                    url_input = gr.Textbox(
                        label="Webpage URL",
                        placeholder="https://en.wikipedia.org/wiki/Computer",
                        lines=1,
                    )
                    url_max = gr.Slider(
                        minimum=1,
                        maximum=15,
                        value=5,
                        step=1,
                        label="Max Images to Caption",
                    )
                    url_btn = gr.Button("Scrape & Caption", variant="primary")

                with gr.Column():
                    url_output = gr.Textbox(
                        label="Captions",
                        lines=15,
                    )

            url_btn.click(
                fn=caption_from_url,
                inputs=[url_input, url_max],
                outputs=url_output,
            )

    gr.Markdown(
        """
        ---
        *Powered by [Hugging Face Transformers](https://huggingface.co/docs/transformers) & [Gradio](https://gradio.app) · 100% open-source, zero API cost.*
        """
    )

# Launch the application
app.launch(share=IS_COLAB)

---

## 6. Engineering Takeaways & Portfolio Notes

### 💡 What I Learned (Useful for LLM / AI Engineering Interviews)

1. **Vision-Language Alignment:** Vision Transformer encoders (ViT) divide an image into fixed patches and convert them into tokens. The language decoder attends to these visual tokens through cross-attention to autoregressively predict the caption text.

2. **Open-Source Model Practicality:** At ~250M parameters, BLIP runs in seconds on consumer CPUs with modest RAM footprints (~1GB), making it ideal for edge deployments and low-cost serving without expensive GPU clusters.

3. **Real-World Web Ingestion:** Building an automated image pipeline requires defensive engineering:
   - Resolving relative URLs via `urllib.parse.urljoin`
   - Filtering out vector SVGs and base64 tracking pixels
   - Enforcing size thresholds to prevent captioning tiny UI icons

4. **Full-Stack Prototyping:** Gradio provides immediate user feedback with tabs, sliders, and public shareable URLs, bridging the gap between raw Python scripts and functional ML prototypes.

---